
### Panpipes h5mu AutoTune

### This Jupyter Notebook can be used to:

### 1. Read an `.h5mu` file.

### 2. Analyze the size and structure of the RNA, PROT, ATAC, and REP modalities.

### 3. Estimate the RAM required for Panpipes preprocessing.

### 4. Automatically calculate the recommended number of CPUs, assuming **15 GB RAM per CPU**.

### 5. Recommend data-driven parameters for Panpipes preprocessing.

### 6. Generate a new YAML configuration file based on the original Panpipes YAML template.

### 7. Generate a parameter recommendation report and a dataset profile.

### **The original `.h5mu` file will not be modified.**


In [ ]:

# =========================
# 0. USER SETTINGS
# =========================

from pathlib import Path

# Set the paths to your input files
H5MU_FILE = Path("/path/to/input.h5mu")
TEMPLATE_YML = Path("/path/to/Preprocess_pipeline_parameter.yml")

# Output files
OUTPUT_YML = Path("Preprocess_pipeline_parameter_recommended.yml")
REPORT_TSV = Path("Panpipes_parameter_recommendations.tsv")
PROFILE_JSON = Path("Panpipes_dataset_profile.json")

# BMRC resource assumption
RAM_PER_CPU_GB = 15

# Preprocessing memory safety factor
# 3.0 = fairly conservative
MEMORY_SAFETY_FACTOR = 3.0

print("H5MU:", H5MU_FILE)
print("Template YAML:", TEMPLATE_YML)


In [ ]:

# =========================
# 1. IMPORTS
# =========================

import os
import json
import math
import copy

import numpy as np
import pandas as pd
import yaml
import mudata as mu

GB = 1024 ** 3


In [ ]:

# =========================
# 2. READ H5MU + YAML
# =========================

if not H5MU_FILE.exists():
    raise FileNotFoundError(H5MU_FILE)

if not TEMPLATE_YML.exists():
    raise FileNotFoundError(TEMPLATE_YML)

with open(TEMPLATE_YML, "r") as f:
    cfg = yaml.safe_load(f)

print("Reading h5mu in backed mode...")
mdata = mu.read_h5mu(H5MU_FILE, backed="r")

print(mdata)
print("\nModalities:", list(mdata.mod.keys()))

for name, ad in mdata.mod.items():
    print(f"{name:8s}: {ad.n_obs:,} cells x {ad.n_vars:,} features")


In [ ]:

# =========================
# 3. HELPER FUNCTIONS
# =========================

def finite_numeric(series):
    x = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    return x[np.isfinite(x)]

def qstats(series):
    x = finite_numeric(series)
    if len(x) == 0:
        return None

    probs = [0, .01, .05, .25, .5, .75, .95, .99, .995, 1]
    vals = np.quantile(x, probs)

    keys = ["min","q1","q5","q25","median","q75","q95","q99","q99.5","max"]
    return {k: float(v) for k, v in zip(keys, vals)}

def round_sensible(x):
    if x is None or not np.isfinite(x):
        return None

    x = float(x)

    if x < 1:
        return round(x, 4)
    elif x < 10:
        return round(x, 2)
    elif x < 100:
        return int(round(x))
    else:
        power = max(0, int(math.floor(math.log10(abs(x)))) - 2)
        base = 10 ** power
        return int(round(x / base) * base)

def robust_lower(series, floor=0):
    x = finite_numeric(series)
    x = x[x >= 0]

    if len(x) < 100:
        return None

    z = np.log1p(x)

    med = np.median(z)
    mad = np.median(np.abs(z - med))
    sigma = 1.4826 * mad

    lower = np.expm1(med - 3.5 * sigma)
    lower = max(lower, floor)

    q05 = np.quantile(x, 0.05)

    # Avoid overly aggressive filtering
    lower = min(lower, q05)

    return round_sensible(lower)

def robust_upper(series):
    x = finite_numeric(series)
    x = x[x >= 0]

    if len(x) < 100:
        return None

    z = np.log1p(x)

    med = np.median(z)
    mad = np.median(np.abs(z - med))
    sigma = 1.4826 * mad

    upper = np.expm1(med + 4.0 * sigma)
    q995 = np.quantile(x, 0.995)

    return round_sensible(max(upper, q995))

def adaptive_percent_upper(series, low_bound, high_bound):
    x = finite_numeric(series)

    if len(x) < 100:
        return None

    q95, q99 = np.quantile(x, [0.95, 0.99])

    value = q95 + 0.5 * (q99 - q95)
    value = max(low_bound, min(high_bound, value))

    return round(float(value), 1)

def get_nested(d, path, default=None):
    cur = d

    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]

    return cur

def set_nested(d, path, value):
    cur = d

    for key in path[:-1]:
        if key not in cur or not isinstance(cur[key], dict):
            cur[key] = {}

        cur = cur[key]

    cur[path[-1]] = value

def choose_color_by(obs):
    for col in ["sample_id", "dataset", "diagnosis", "batch"]:
        if col in obs.columns and obs[col].nunique(dropna=True) > 1:
            return col

    return None

def choose_grouping_var(obs):
    preferred = [
        "sample_id",
        "dataset",
        "treatment",
        "inflammation_status",
        "diagnosis",
        "batch"
    ]

    chosen = []

    for col in preferred:
        if col in obs.columns:
            n = obs[col].nunique(dropna=True)

            if 1 < n <= 100:
                chosen.append(col)

    return ",".join(chosen[:6]) if chosen else None

def choose_hvg_batch_key(obs):
    for col in ["sample_id", "dataset", "library_id", "batch"]:
        if col in obs.columns:
            n = obs[col].nunique(dropna=True)

            if 1 < n <= 500:
                return col

    return None

def suggest_hvg_n(n_cells, n_genes):
    if n_genes <= 3000:
        return max(1000, min(2000, n_genes - 1))

    if n_cells < 10_000:
        return 2000

    if n_cells < 100_000:
        return 2500

    return min(3000, n_genes - 1)

def suggest_pcs(n_cells, n_genes, n_hvg):
    if n_cells < 5_000:
        pcs = 30
    elif n_cells < 50_000:
        pcs = 40
    else:
        pcs = 50

    return int(
        max(
            10,
            min(
                pcs,
                n_hvg - 1,
                n_genes - 1
            )
        )
    )


In [ ]:

# =========================
# 4. MEMORY ESTIMATION
# =========================

def dtype_itemsize(dtype, default=4):
    try:
        return np.dtype(dtype).itemsize
    except Exception:
        return default

def estimate_density(ad, max_rows=1000, max_cols=2000):
    nr = min(max_rows, ad.n_obs)
    nc = min(max_cols, ad.n_vars)

    if nr == 0 or nc == 0:
        return 0

    try:
        x = ad.X[:nr, :nc]

        if hasattr(x, "nnz"):
            return x.nnz / (nr * nc)

        arr = np.asarray(x)
        return np.count_nonzero(arr) / arr.size

    except Exception:
        return None

def estimate_matrix_bytes(ad):
    n_obs = int(ad.n_obs)
    n_vars = int(ad.n_vars)

    try:
        itemsize = dtype_itemsize(ad.X.dtype, 4)
    except Exception:
        itemsize = 4

    # sparse matrix
    try:
        if hasattr(ad.X, "nnz"):
            nnz = int(ad.X.nnz)

            data_bytes = nnz * itemsize
            indices_bytes = nnz * 4
            indptr_bytes = (n_obs + 1) * 8

            return data_bytes + indices_bytes + indptr_bytes

    except Exception:
        pass

    density = estimate_density(ad)

    # If the sampled matrix appears sparse
    if density is not None and density < 0.3:

        nnz_est = int(n_obs * n_vars * density)

        return (
            nnz_est * (itemsize + 4)
            + (n_obs + 1) * 8
        )

    # dense fallback
    return n_obs * n_vars * itemsize

def estimate_dataframe_bytes(df):
    try:
        return int(
            df.memory_usage(
                index=True,
                deep=True
            ).sum()
        )
    except Exception:
        return 0

def estimate_base_memory(mdata):
    total = 0
    details = {}

    for mod_name, ad in mdata.mod.items():

        x_bytes = estimate_matrix_bytes(ad)

        obs_bytes = estimate_dataframe_bytes(ad.obs)
        var_bytes = estimate_dataframe_bytes(ad.var)

        layers_bytes = 0

        for lname in ad.layers.keys():

            layer = ad.layers[lname]

            try:
                if hasattr(layer, "nnz"):

                    nnz = int(layer.nnz)
                    itemsize = dtype_itemsize(layer.dtype, 4)

                    b = (
                        nnz * (itemsize + 4)
                        + (ad.n_obs + 1) * 8
                    )

                else:

                    b = (
                        int(np.prod(layer.shape))
                        * dtype_itemsize(layer.dtype, 4)
                    )

            except Exception:
                b = 0

            layers_bytes += b

        mod_total = (
            x_bytes
            + obs_bytes
            + var_bytes
            + layers_bytes
        )

        total += mod_total

        details[mod_name] = {
            "n_obs": int(ad.n_obs),
            "n_vars": int(ad.n_vars),
            "X_GB": round(x_bytes / GB, 3),
            "layers_GB": round(layers_bytes / GB, 3),
            "metadata_GB": round((obs_bytes + var_bytes) / GB, 3),
            "estimated_total_GB": round(mod_total / GB, 3),
        }

    return total, details


disk_bytes = H5MU_FILE.stat().st_size

base_memory_bytes, modality_memory = estimate_base_memory(mdata)

# HDF5 may be compressed, so use at least:
# max(estimated in-memory size, file size x 2)
lower_bound_bytes = max(
    base_memory_bytes,
    disk_bytes * 2
)

recommended_ram_bytes = (
    lower_bound_bytes
    * MEMORY_SAFETY_FACTOR
    + 8 * GB
)

recommended_ram_gb = recommended_ram_bytes / GB

recommended_cpu = math.ceil(
    recommended_ram_gb
    / RAM_PER_CPU_GB
)

threads_high = min(
    recommended_cpu,
    40
)

threads_medium = max(
    2,
    min(
        20,
        math.ceil(threads_high / 2)
    )
)

threads_low = max(
    1,
    min(
        10,
        math.ceil(threads_high / 4)
    )
)

print("===== MEMORY / CPU RECOMMENDATION =====")
print(f"H5MU disk size         : {disk_bytes / GB:.2f} GB")
print(f"Estimated base memory  : {base_memory_bytes / GB:.2f} GB")
print(f"Recommended RAM        : {recommended_ram_gb:.2f} GB")
print(f"RAM per CPU            : {RAM_PER_CPU_GB} GB")
print(f"Recommended CPU        : {recommended_cpu}")
print()
print(f"threads_high           : {threads_high}")
print(f"threads_medium         : {threads_medium}")
print(f"threads_low            : {threads_low}")
print()
print("Per-modality estimate:")
display(pd.DataFrame(modality_memory).T)


In [ ]:

# =========================
# 5. DOUBLET THRESHOLD
# =========================

def best_threshold_from_labels(scores, labels):

    scores = np.asarray(scores, dtype=float)

    labels = pd.Series(labels)

    mask = (
        np.isfinite(scores)
        & labels.notna().to_numpy()
    )

    scores = scores[mask]
    labels = labels[mask]

    if len(scores) < 100:
        return None

    y = labels.map(
        lambda x:
            x
            if isinstance(x, (bool, np.bool_))
            else str(x).lower() in {
                "true",
                "1",
                "doublet"
            }
    ).to_numpy(bool)

    if y.sum() == 0 or (~y).sum() == 0:
        return None

    candidates = np.unique(
        np.quantile(
            scores,
            np.linspace(
                0.01,
                0.99,
                300
            )
        )
    )

    best_threshold = None
    best_balacc = -1

    for threshold in candidates:

        pred = scores >= threshold

        tpr = (
            (pred & y).sum()
            / max(1, y.sum())
        )

        tnr = (
            ((~pred) & (~y)).sum()
            / max(1, (~y).sum())
        )

        balacc = (
            tpr
            + tnr
        ) / 2

        if balacc > best_balacc:

            best_balacc = balacc
            best_threshold = threshold

    return float(best_threshold)

def recommend_doublet_threshold(obs):

    if "doublet_scores" not in obs.columns:
        return None, "doublet_scores absent"

    threshold = None

    if "is_doublet" in obs.columns:

        threshold = best_threshold_from_labels(
            obs["doublet_scores"],
            obs["is_doublet"]
        )

        if threshold is not None:
            return (
                round(threshold, 4),
                "optimized using is_doublet labels"
            )

    scores = finite_numeric(
        obs["doublet_scores"]
    )

    if len(scores) < 500:
        return None, "insufficient doublet scores"

    # Fallback: top 6%
    threshold = np.quantile(
        scores,
        0.94
    )

    return (
        round(float(threshold), 4),
        "fallback: top 6% empirical doublet-score tail"
    )


In [ ]:

# =========================
# 6. ANALYSE DATASET
# =========================

mods = set(
    mdata.mod.keys()
)

rna = (
    mdata.mod["rna"]
    if "rna" in mods
    else None
)

prot = (
    mdata.mod["prot"]
    if "prot" in mods
    else None
)

atac = (
    mdata.mod["atac"]
    if "atac" in mods
    else None
)

rep = (
    mdata.mod["rep"]
    if "rep" in mods
    else None
)


profile = {
    "file": str(H5MU_FILE),
    "disk_size_GB": disk_bytes / GB,
    "estimated_base_memory_GB": base_memory_bytes / GB,
    "recommended_RAM_GB": recommended_ram_gb,
    "RAM_per_CPU_GB": RAM_PER_CPU_GB,
    "recommended_CPU": recommended_cpu,
    "threads_high": threads_high,
    "threads_medium": threads_medium,
    "threads_low": threads_low,
    "modalities": list(mods),
    "modality_memory": modality_memory,
}


if rna is not None:

    profile["rna"] = {
        "n_cells": int(rna.n_obs),
        "n_genes": int(rna.n_vars),
        "obs_columns": list(rna.obs.columns),
        "qc": {}
    }

    for metric in [
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt",
        "pct_counts_rp",
        "doublet_scores"
    ]:
        if metric in rna.obs.columns:
            profile["rna"]["qc"][metric] = qstats(
                rna.obs[metric]
            )


if prot is not None:

    profile["prot"] = {
        "n_cells": int(prot.n_obs),
        "n_features": int(prot.n_vars),
        "obs_columns": list(prot.obs.columns),
        "qc": {}
    }

    for metric in [
        "total_counts",
        "pct_counts_isotype",
        "n_prot_by_counts"
    ]:
        if metric in prot.obs.columns:
            profile["prot"]["qc"][metric] = qstats(
                prot.obs[metric]
            )

print(json.dumps(
    {
        k: v
        for k, v in profile.items()
        if k not in ["rna","prot"]
    },
    indent=2
))


In [ ]:

# =========================
# 7. BUILD RECOMMENDED YAML
# =========================

new_cfg = copy.deepcopy(cfg)

recommendations = []

def rec(path, value, reason):

    old = get_nested(
        cfg,
        path
    )

    set_nested(
        new_cfg,
        path,
        value
    )

    recommendations.append({
        "parameter": ".".join(path),
        "original": old,
        "recommended": value,
        "changed": old != value,
        "reason": reason
    })


# ---------- resources ----------

rec(
    ["resources","threads_high"],
    threads_high,
    "calculated from RAM requirement / 15 GB per CPU"
)

rec(
    ["resources","threads_medium"],
    threads_medium,
    "medium resource tier"
)

rec(
    ["resources","threads_low"],
    threads_low,
    "low resource tier"
)


# ---------- input ----------

rec(
    ["unfiltered_obj"],
    str(H5MU_FILE),
    "use profiled h5mu"
)


# ---------- modalities ----------

for mod in [
    "rna",
    "prot",
    "rep",
    "atac"
]:

    rec(
        ["modalities",mod],
        mod in mods,
        f"detected modalities: {sorted(mods)}"
    )


# ---------- RNA ----------

if rna is not None:

    obs = rna.obs


    if "n_genes_by_counts" in obs.columns:

        rec(
            [
                "filtering",
                "rna",
                "obs",
                "min",
                "n_genes_by_counts"
            ],
            robust_lower(
                obs["n_genes_by_counts"],
                floor=200
            ),
            "data-driven lower cutoff"
        )


    if "total_counts" in obs.columns:

        rec(
            [
                "filtering",
                "rna",
                "obs",
                "min",
                "total_counts"
            ],
            robust_lower(
                obs["total_counts"],
                floor=200
            ),
            "data-driven lower cutoff"
        )

        rec(
            [
                "filtering",
                "rna",
                "obs",
                "max",
                "total_counts"
            ],
            robust_upper(
                obs["total_counts"]
            ),
            "data-driven upper cutoff"
        )


    if "pct_counts_mt" in obs.columns:

        rec(
            [
                "filtering",
                "rna",
                "obs",
                "max",
                "pct_counts_mt"
            ],
            adaptive_percent_upper(
                obs["pct_counts_mt"],
                10,
                30
            ),
            "data-driven mitochondrial cutoff"
        )


    if "pct_counts_rp" in obs.columns:

        rec(
            [
                "filtering",
                "rna",
                "obs",
                "max",
                "pct_counts_rp"
            ],
            adaptive_percent_upper(
                obs["pct_counts_rp"],
                30,
                80
            ),
            "data-driven ribosomal cutoff"
        )


    doublet_thr, doublet_reason = (
        recommend_doublet_threshold(
            obs
        )
    )

    rec(
        [
            "filtering",
            "rna",
            "obs",
            "max",
            "doublet_scores"
        ],
        doublet_thr,
        doublet_reason
    )


    # HVG

    n_hvg = suggest_hvg_n(
        rna.n_obs,
        rna.n_vars
    )

    rec(
        ["hvg","batch_key"],
        choose_hvg_batch_key(obs),
        "automatic batch key"
    )

    rec(
        ["hvg","n_top_genes"],
        n_hvg,
        "based on dataset size"
    )

    rec(
        ["hvg","flavor"],
        "seurat",
        "compatible with Panpipes normalize_total + log1p"
    )

    rec(
        ["hvg","filter"],
        False,
        "retain all genes"
    )


    # PCA

    rec(
        ["pca","n_pcs"],
        suggest_pcs(
            rna.n_obs,
            rna.n_vars,
            n_hvg
        ),
        "based on dataset size"
    )

    rec(
        ["pca","color_by"],
        choose_color_by(obs),
        "automatic plotting variable"
    )


    # QC plot

    rec(
        ["plotqc","grouping_var"],
        choose_grouping_var(obs),
        "use informative columns present in obs"
    )

    rna_metrics = [
        x
        for x in [
            "pct_counts_mt",
            "pct_counts_rp",
            "pct_counts_hb",
            "pct_counts_ig",
            "doublet_scores"
        ]
        if x in obs.columns
    ]

    rec(
        ["plotqc","rna_metrics"],
        ",".join(rna_metrics),
        "only existing RNA QC metrics"
    )


# ---------- Protein ----------

if prot is not None:

    obs = prot.obs


    if "total_counts" in obs.columns:

        rec(
            [
                "filtering",
                "prot",
                "obs",
                "max",
                "total_counts"
            ],
            robust_upper(
                obs["total_counts"]
            ),
            "data-driven protein cutoff"
        )


    if "pct_counts_isotype" in obs.columns:

        rec(
            [
                "filtering",
                "prot",
                "obs",
                "max",
                "pct_counts_isotype"
            ],
            adaptive_percent_upper(
                obs["pct_counts_isotype"],
                3,
                20
            ),
            "data-driven isotype cutoff"
        )


    background = get_nested(
        cfg,
        ["prot","background_obj"]
    )

    background_exists = (
        bool(background)
        and Path(str(background)).exists()
    )


    rec(
        ["prot","normalisation_methods"],
        "clr,dsb" if background_exists else "clr",
        "DSB only when background object exists"
    )


    rec(
        ["prot","clr_margin"],
        1,
        "feature-wise CLR"
    )


    rec(
        ["prot","store_as_X"],
        "dsb" if background_exists else "clr",
        "store valid normalized protein matrix"
    )


    rec(
        ["prot","n_pcs"],
        int(
            max(
                2,
                min(
                    50,
                    prot.n_vars - 1
                )
            )
        ),
        "bounded by number of proteins"
    )


    rec(
        ["prot","color_by"],
        choose_color_by(obs),
        "automatic plotting variable"
    )


# ---------- intersect modalities ----------

active_mods = [
    m
    for m in [
        "rna",
        "prot",
        "atac",
        "rep"
    ]
    if m in mods
]

if len(active_mods) > 1:

    barcode_sets = {
        m: set(
            map(
                str,
                mdata.mod[m].obs_names
            )
        )
        for m in active_mods
    }

    common = set.intersection(
        *barcode_sets.values()
    )

    smallest = min(
        len(x)
        for x in barcode_sets.values()
    )

    overlap = (
        len(common)
        / smallest
    )

    if overlap >= 0.95:

        intersect_value = ",".join(
            active_mods
        )

    else:

        intersect_value = None

else:

    intersect_value = None


rec(
    ["intersect_mods"],
    intersect_value,
    "automatic barcode overlap check"
)


report_df = pd.DataFrame(
    recommendations
)

display(
    report_df[
        report_df["changed"]
    ].reset_index(drop=True)
)


In [ ]:

# =========================
# 8. WRITE NEW YAML
# =========================

header = (
    "# AUTO-GENERATED Panpipes preprocess recommendation\n"
    f"# Input h5mu: {H5MU_FILE}\n"
    f"# H5MU disk size: {disk_bytes / GB:.2f} GB\n"
    f"# Estimated preprocess RAM: {recommended_ram_gb:.2f} GB\n"
    f"# RAM per CPU assumption: {RAM_PER_CPU_GB} GB\n"
    f"# Recommended CPU: {recommended_cpu}\n"
    "\n"
)

with open(
    OUTPUT_YML,
    "w"
) as f:

    f.write(header)

    yaml.safe_dump(
        new_cfg,
        f,
        sort_keys=False,
        allow_unicode=True
    )


report_df.to_csv(
    REPORT_TSV,
    sep="\t",
    index=False
)


with open(
    PROFILE_JSON,
    "w"
) as f:

    json.dump(
        profile,
        f,
        indent=2
    )


print("DONE")
print()
print("Recommended YAML:")
print(OUTPUT_YML.resolve())
print()
print("Parameter report:")
print(REPORT_TSV.resolve())
print()
print("Dataset profile:")
print(PROFILE_JSON.resolve())


In [ ]:

# =========================
# 9. PREVIEW FINAL YAML
# =========================

print(
    OUTPUT_YML.read_text()
)


## Output Files

# After the notebook has finished running, the following files will be generated:

# ```text
# Preprocess_pipeline_parameter_recommended.yml
# Panpipes_parameter_recommendations.tsv
# Panpipes_dataset_profile.json


# * Preprocess_pipeline_parameter_recommended.yml
    # The recommended YAML configuration file for running Panpipes preprocessing.
# * Panpipes_parameter_recommendations.tsv
    # A report showing the original and recommended parameter values, whether each parameter was changed, and the reason for the recommendation.
# * Panpipes_dataset_profile.json
    # A summary of the input dataset, including file size, estimated RAM requirements, recommended CPU resources, modality dimensions, and QC statistics.